<a href="https://colab.research.google.com/github/FELIPEACASTRO/KG1-NVIDIA/blob/claude/competent-shamir/notebooks/KG1_v15_TONG_REPLICA_COLAB.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# KG1 v15 — TONG REPLICA (Colab Pro H100)

Baseline técnico **replicando Tong Hui Kang** (LB 0.85 x3, LB 0.84 x5) adaptado para Colab.

## Evidência pública validada (audit 2026-04-19)
- Top LB atual: **0.86** (178 teams)
- Tong maximum: **LB 0.85 x3, 0.84 x5**
- Taha (tahaalam2009): **90.7% TRAIN SET** (rodando Tong's solvers, NÃO é LB)
- Galliano gates taxonomy: **NÃO validada empiricamente** (ele se retirou)
- CPMP host: **Gemini Flash 2.0 distillation APROVADO** (não só DeepSeek)

## Configuração Tong (replicada)
| Config | Valor | Fonte |
|---|---|---|
| Base model | `nvidia/NVIDIA-Nemotron-3-Nano-30B-A3B-BF16` | Tong writeup |
| LoRA rank | 32 | Tong |
| **train_unembed** | **True** (lm_head in LoRA) | Tong (KEY) |
| LR | **2e-4** | Tong |
| LR schedule | **StepLinearDecay** | Tong |
| Batch / micro_batch | 64 / 16 (effective 64) | Tong |
| adam beta1/beta2 | 0.9 / 0.95 | Tong |
| weight_decay | 0 | Tong |
| max_length | 8192 | Tong |
| Epochs | 1 | Tong |
| Loss target | **min_logprob > -0.69** (approximated) | Tong KEY INNOVATION |
| Stratified batches | True (per category) | Tong |

## V15 predicted LB (4-API consensus)
| API | Floor | Median | Ceiling |
|---|---|---|---|
| GPT-5.4 | 0.76 | 0.80-0.82 | 0.84 |
| Claude-Opus-4-7 | 0.72 | 0.755 | 0.79 |
| DeepSeek-R1 | 0.82 | 0.835-0.855 | 0.87 |
| GPT-5.4-pro | 0.82 | 0.845 | 0.87 |
| **CONSENSUS** | **0.78** | **0.82** | **0.86** |

**NOT 0.90** — that target is fantasy per all 4 APIs. V15 is the BASELINE step.

## Credenciais (Colab Secrets)

| Secret | Value |
|---|---|
| `HF_KEY` | seu token HF |
| `KAGGLE_USERNAME` | `felipe1983` |
| `KAGGLE_KEY` | `93dbcf741dba9085eded2cdbe2fc0cab` |

## Cell 1 — Setup Colab + secrets

In [ ]:
import os, sys, subprocess, json, time, math

!nvidia-smi --query-gpu=name,memory.total,memory.free --format=csv,noheader
print('Python: ' + sys.version.split()[0])

from google.colab import userdata
os.environ['HF_TOKEN'] = userdata.get('HF_KEY')
os.environ['KAGGLE_USERNAME'] = userdata.get('KAGGLE_USERNAME')
os.environ['KAGGLE_KEY'] = userdata.get('KAGGLE_KEY')

assert os.environ.get('HF_TOKEN', '').startswith('hf_')
assert os.environ.get('KAGGLE_USERNAME')

from huggingface_hub import whoami
info = whoami(token=os.environ['HF_TOKEN'])
print('HF User: ' + info['name'])

import torch
gpu = torch.cuda.get_device_name(0)
mem = torch.cuda.get_device_properties(0).total_memory / 1e9
print(f'GPU: {gpu} {mem:.0f} GB')
assert mem >= 75, 'Need H100 80GB High-RAM'

# Keep-alive
from IPython.display import display, Javascript
display(Javascript("setInterval(() => { "
    "document.querySelector('colab-toolbar-button#connect')?.click(); "
    "}, 60000);"))

## Cell 2 — Google Drive + resume

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

GDRIVE_BASE = '/content/drive/MyDrive/KG1_v15_TONG'
LOCAL_BASE = '/content/kg1'
CHECKPOINT_DIR = GDRIVE_BASE + '/checkpoints'
SUBMISSIONS_DIR = GDRIVE_BASE + '/submissions'
LOGS_DIR = GDRIVE_BASE + '/logs'
for d in [GDRIVE_BASE, CHECKPOINT_DIR, SUBMISSIONS_DIR, LOGS_DIR, LOCAL_BASE]:
    os.makedirs(d, exist_ok=True)

import glob
existing = sorted(glob.glob(CHECKPOINT_DIR + '/checkpoint-*'),
                  key=lambda p: int(p.rsplit('-', 1)[-1]) if '-' in p else 0)
RESUME_FROM = int(existing[-1].rsplit('-', 1)[-1]) if existing else 0
print(f'Resume: step {RESUME_FROM}' if RESUME_FROM else '[FRESH RUN]')

## Cell 3 — Install deps (Tong-compatible)

In [ ]:
import subprocess
subprocess.run(['pip', 'uninstall', '-y', 'torchao'], capture_output=True, text=True)

!pip install -q \
    "transformers==4.55.0" \
    "tokenizers==0.21.0" \
    "huggingface_hub>=0.34,<1.0" \
    "peft>=0.14" \
    "bitsandbytes>=0.44" \
    "accelerate>=1.7" \
    "datasets>=3.0" \
    "safetensors>=0.5" \
    "kaggle>=1.6" \
    "trl>=0.12" \
    "protobuf>=4.25,<5.0" \
    --force-reinstall --no-deps 2>&1 | tail -3

WHEELS = [
    'https://github.com/state-spaces/mamba/releases/download/v2.3.1/mamba_ssm-2.3.1+cu12torch2.10cxx11abiTRUE-cp312-cp312-linux_x86_64.whl',
    'https://github.com/Dao-AILab/causal-conv1d/releases/download/v1.6.1.post4/causal_conv1d-1.6.1+cu12torch2.10cxx11abiTRUE-cp312-cp312-linux_x86_64.whl',
]
for u in WHEELS:
    subprocess.run(['pip', 'install', '-q', u, '--no-deps'], capture_output=True, timeout=180)

for m in list(sys.modules.keys()):
    if any(m.startswith(p) for p in ['transformers','tokenizers','huggingface','peft','mamba_ssm','causal_conv1d']):
        del sys.modules[m]

import transformers, peft, bitsandbytes, mamba_ssm, causal_conv1d
print(f'transformers: {transformers.__version__}')
print(f'peft: {peft.__version__}')
print(f'mamba_ssm: {mamba_ssm.__version__}')
assert transformers.__version__.startswith('4.55')

## Cell 4 — Clone Tong repo (for investigators/solvers)

In [ ]:
import subprocess
# Clone Tong's open-source Progress Prize submission
tong_dir = LOCAL_BASE + '/tonghuikang_nemotron'
if not os.path.exists(tong_dir):
    subprocess.run(['git', 'clone', 'https://github.com/tonghuikang/nemotron.git', tong_dir],
                   capture_output=True, timeout=120)
else:
    subprocess.run(['git', '-C', tong_dir, 'pull'], capture_output=True, timeout=60)

print('Tong repo files:')
for f in sorted(os.listdir(tong_dir))[:20]:
    print(f'  {f}')

print('\nInvestigators (family solvers):')
for f in sorted(os.listdir(tong_dir + '/investigators')):
    print(f'  {f}')

sys.path.insert(0, tong_dir)
sys.path.insert(0, tong_dir + '/investigators')
# These import Tong's solvers for bit_manipulation, cryptarithm_deduce, etc

## Cell 5 — Download Kaggle train.csv

In [ ]:
os.makedirs(os.path.expanduser('~/.kaggle'), exist_ok=True)
with open(os.path.expanduser('~/.kaggle/kaggle.json'), 'w') as f:
    json.dump({'username': os.environ['KAGGLE_USERNAME'], 'key': os.environ['KAGGLE_KEY']}, f)
os.chmod(os.path.expanduser('~/.kaggle/kaggle.json'), 0o600)

train_dir = LOCAL_BASE + '/kaggle_data'
if not os.path.exists(train_dir + '/train.csv'):
    os.makedirs(train_dir, exist_ok=True)
    subprocess.run(
        ['kaggle', 'competitions', 'download',
         '-c', 'nvidia-nemotron-model-reasoning-challenge',
         '-p', train_dir, '-f', 'train.csv'],
        capture_output=True, timeout=300
    )
    import zipfile
    for z in glob.glob(train_dir + '/*.zip'):
        with zipfile.ZipFile(z) as zf: zf.extractall(train_dir)

import pandas as pd
df = pd.read_csv(train_dir + '/train.csv')
print(f'train.csv: {len(df)} rows, columns: {list(df.columns)}')
print(df.head(2).to_dict())

# Per-family distribution
if 'category' in df.columns:
    print('\nCategory counts:')
    print(df['category'].value_counts())

## Cell 6 — Load Nemotron-30B NF4 + skip_modules

In [ ]:
# Patches (V13-proven)
import transformers.utils, transformers.utils.import_utils
transformers.utils.is_torch_flex_attn_available = lambda: False
transformers.utils.import_utils.is_torch_flex_attn_available = lambda: False
from huggingface_hub import HfApi
_orig = HfApi.list_repo_tree
def _safe(self, *a, **k):
    try: return list(_orig(self, *a, **k))
    except Exception as e:
        if '404' in str(e) or 'Not Found' in str(e): return []
        raise
HfApi.list_repo_tree = _safe
import transformers.utils.hub as tuh, transformers.tokenization_utils_base as tub
tuh.list_repo_templates = lambda *a, **k: []
tub.list_repo_templates = lambda *a, **k: []
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'

from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
MODEL = 'nvidia/NVIDIA-Nemotron-3-Nano-30B-A3B-BF16'
HF_TOKEN = os.environ['HF_TOKEN']

tokenizer = AutoTokenizer.from_pretrained(MODEL, trust_remote_code=True, token=HF_TOKEN)
if tokenizer.pad_token is None: tokenizer.pad_token = tokenizer.eos_token
print(f'Tokenizer vocab: {len(tokenizer)}')

bnb = BitsAndBytesConfig(
    load_in_4bit=True, bnb_4bit_quant_type='nf4',
    bnb_4bit_use_double_quant=True, bnb_4bit_compute_dtype=torch.bfloat16,
    llm_int8_skip_modules=['out_proj', 'lm_head'],  # Mamba+embedding safety
)

t0 = time.time()
model = AutoModelForCausalLM.from_pretrained(
    MODEL, torch_dtype=torch.bfloat16, device_map='auto',
    trust_remote_code=True, attn_implementation='sdpa',
    quantization_config=bnb, token=HF_TOKEN,
)
# Disable router aux_loss (frozen router)
if hasattr(model.config, 'output_router_logits'):
    model.config.output_router_logits = False
if hasattr(model.config, 'router_aux_loss_coef'):
    model.config.router_aux_loss_coef = 0.0
print(f'Loaded NF4 in {(time.time()-t0)/60:.1f}min | VRAM: {torch.cuda.memory_allocated()/1e9:.1f} GB')

## Cell 7 — Tong's LoRA (9 targets incl. lm_head = train_unembed)

In [ ]:
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

model = prepare_model_for_kbit_training(model, use_gradient_checkpointing=True)
model.gradient_checkpointing_enable()
model.enable_input_require_grads()

# Tong's targets: attention + mamba + MLP + lm_head (train_unembed=True)
TONG_TARGETS = ['q_proj', 'k_proj', 'v_proj', 'o_proj',
                'in_proj', 'out_proj',  # Mamba (LoRA on BF16 via skip_modules)
                'gate_proj', 'up_proj', 'down_proj',  # MLP
                'lm_head']  # train_unembed=True (Tong KEY)

lc = LoraConfig(r=32, lora_alpha=32, lora_dropout=0.0,
                target_modules=TONG_TARGETS,
                bias='none', task_type='CAUSAL_LM')
model = get_peft_model(model, lc)

trnb = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())
print(f'Trainable: {trnb:,} ({100*trnb/total:.2f}%)')
print(f'VRAM post-LoRA: {torch.cuda.memory_allocated()/1e9:.1f} GB')

## Cell 8 — Generate training data (Tong's reasoning pipeline)

In [ ]:
# NOTE: Tong's full pipeline requires Tinker+Modal ($212). Here we approximate:
# 1. Use train.csv directly
# 2. Stratified sampling per category
# 3. Apply Tong's CoT format

import random
random.seed(42)

PROMPT_SUFFIX = '\nPlease put your final answer inside `\\boxed{}`. For example: `\\boxed{your answer}`'

records = []
for _, row in df.iterrows():
    category = row.get('category', 'unknown')
    prompt = str(row.get('question', row.get('prompt', '')))
    answer = str(row.get('answer', ''))
    if not prompt or not answer: continue
    cot = str(row.get('cot', row.get('chain_of_thought', '')))
    if not cot: cot = f'Let me solve this step by step.\n\nThe answer is \\boxed{{{answer}}}'

    # Tong format: messages with <think>...</think>\\boxed{answer}
    if '\\boxed{' not in cot:
        cot += f'\n\n\\boxed{{{answer}}}'

    records.append({
        'category': category,
        'messages': [
            {'role': 'user', 'content': prompt + PROMPT_SUFFIX},
            {'role': 'assistant', 'content': cot},
        ],
    })

print(f'Total records: {len(records)}')

# Stratified sample per category (Tong's approach)
from collections import defaultdict
by_cat = defaultdict(list)
for r in records: by_cat[r['category']].append(r)
for c, rs in by_cat.items():
    print(f'  {c}: {len(rs)}')

# Take equal samples per category
TARGET_PER_CAT = 800
balanced = []
for c, rs in by_cat.items():
    random.shuffle(rs)
    balanced.extend(rs[:TARGET_PER_CAT])
random.shuffle(balanced)
print(f'Balanced dataset: {len(balanced)}')

## Cell 9 — Tokenize (max_length=4096 for H100 safety)

In [ ]:
MAX_LENGTH = 4096  # Tong uses 8192 but H100 80GB + NF4 = 4096 safer

def tokenize(ex):
    try:
        full = tokenizer.apply_chat_template(
            ex['messages'], tokenize=False, add_generation_prompt=False,
            enable_thinking=True,  # Tong uses this
        )
    except TypeError:
        full = tokenizer.apply_chat_template(ex['messages'], tokenize=False, add_generation_prompt=False)
    ids = tokenizer.encode(full, add_special_tokens=False)
    prompt_msgs = [m for m in ex['messages'] if m['role'] != 'assistant']
    try:
        pt = tokenizer.apply_chat_template(prompt_msgs, tokenize=False, add_generation_prompt=True, enable_thinking=True)
    except TypeError:
        pt = tokenizer.apply_chat_template(prompt_msgs, tokenize=False, add_generation_prompt=True)
    pids = tokenizer.encode(pt, add_special_tokens=False)
    pl = min(len(pids), len(ids))
    mask = [0]*pl + [1]*(len(ids)-pl)
    if len(ids) > MAX_LENGTH:
        ids = ids[:MAX_LENGTH]; mask = mask[:MAX_LENGTH]
    return {'input_ids': ids, 'loss_mask': mask, 'category': ex['category']}

train_data = []
for ex in balanced:
    try:
        t = tokenize(ex)
        if sum(t['loss_mask']) > 0: train_data.append(t)
    except Exception: pass
print(f'Tokenized: {len(train_data)}')

## Cell 10 — Train (Tong config, min-logprob approx)

In [ ]:
import zipfile, hashlib, shutil

# Tong's exact hyperparams
LR = 2e-4
BATCH = 8
GRAD_ACCUM = 8  # Effective batch = 64 (Tong's value)
MAX_STEPS = 445  # Tong's 27.85M tokens / ~62k per optim step
MAX_GRAD_NORM = 1.0  # V14.2 safety (Tong uses 1e9 = disabled)

opt = torch.optim.AdamW(
    [p for p in model.parameters() if p.requires_grad],
    lr=LR, betas=(0.9, 0.95), eps=1e-8, weight_decay=0.0,
)

def lr_at(s):  # Tong's StepLinearDecay
    return LR * max(0.0, 1 - s / MAX_STEPS)

def collate(b):
    ml = max(len(x['input_ids']) for x in b)
    pad = tokenizer.pad_token_id
    ids = [x['input_ids'] + [pad]*(ml-len(x['input_ids'])) for x in b]
    att = [[1]*len(x['input_ids']) + [0]*(ml-len(x['input_ids'])) for x in b]
    lmk = [x['loss_mask'] + [0]*(ml-len(x['loss_mask'])) for x in b]
    return {
        'input_ids': torch.tensor(ids, dtype=torch.long).cuda(),
        'attention_mask': torch.tensor(att, dtype=torch.long).cuda(),
        'loss_mask': torch.tensor(lmk, dtype=torch.float).cuda(),
    }

def compute_loss_min_logprob_aware(logits, labels, mask):
    """Tong's key innovation: penalize WORST token more than avg."""
    # Standard next-token CE
    shift_logits = logits[..., :-1, :].contiguous()
    shift_labels = labels[..., 1:].contiguous()
    shift_mask = mask[..., 1:].contiguous()
    loss_fct = torch.nn.CrossEntropyLoss(reduction='none')
    per_token = loss_fct(
        shift_logits.view(-1, shift_logits.size(-1)),
        shift_labels.view(-1),
    ).view(shift_labels.shape)
    # Apply mask
    masked = per_token * shift_mask
    n_valid = shift_mask.sum().clamp(min=1)
    # Tong approximation: avg + alpha * max  (boosts min-logprob signal)
    avg_loss = masked.sum() / n_valid
    # Max token loss (excluding masked-out)
    max_masked = (per_token * shift_mask + (1 - shift_mask) * -1e9)
    max_loss = max_masked.max()
    # Weighted sum: 0.7 avg + 0.3 max (approximates min-logprob target)
    return 0.7 * avg_loss + 0.3 * max_loss, n_valid

# Stratified batches per category (Tong's approach)
def make_stratified_order(data, batch_size):
    from collections import defaultdict
    by_cat = defaultdict(list)
    for i, x in enumerate(data): by_cat[x['category']].append(i)
    cats = list(by_cat.keys())
    random.seed(42)
    for v in by_cat.values(): random.shuffle(v)
    result = []
    cursors = {c: 0 for c in cats}
    while True:
        batch = []
        for c in cats:
            if cursors[c] < len(by_cat[c]):
                batch.append(by_cat[c][cursors[c]])
                cursors[c] += 1
            if len(batch) >= batch_size: break
        if len(batch) < batch_size: break
        result.extend(batch)
    return [data[i] for i in result]

ordered_data = make_stratified_order(train_data, BATCH)
print(f'Ordered: {len(ordered_data)}')

model.train()
gs = 0
start = time.time()
for ss in range(0, len(ordered_data), BATCH * GRAD_ACCUM):
    if gs >= MAX_STEPS: break
    for pg in opt.param_groups: pg['lr'] = lr_at(gs)
    opt.zero_grad()
    total_loss = 0.0
    for a in range(GRAD_ACCUM):
        chunk = ordered_data[ss + a*BATCH : ss + (a+1)*BATCH]
        if len(chunk) < BATCH: continue
        mb = collate(chunk)
        out = model(input_ids=mb['input_ids'], attention_mask=mb['attention_mask'])
        loss, n = compute_loss_min_logprob_aware(out.logits, mb['input_ids'], mb['loss_mask'])
        (loss / GRAD_ACCUM).backward()
        total_loss += loss.item()
    torch.nn.utils.clip_grad_norm_(
        [p for p in model.parameters() if p.requires_grad], MAX_GRAD_NORM)
    opt.step()
    gs += 1
    if gs % 10 == 0:
        torch.cuda.empty_cache()
        print(f'step {gs}/{MAX_STEPS} | avg_loss {total_loss/GRAD_ACCUM:.4f} | vram {torch.cuda.memory_reserved()/1e9:.1f}GB | {(time.time()-start)/60:.1f}m')
        sys.stdout.flush()
    if gs % 100 == 0:
        ckpt = CHECKPOINT_DIR + f'/checkpoint-{gs}'
        os.makedirs(ckpt, exist_ok=True)
        model.save_pretrained(ckpt)
        print(f'[CKPT SAVED] {ckpt}')

model.save_pretrained(CHECKPOINT_DIR + '/final')
print(f'Training done. Steps: {gs}. Total time: {(time.time()-start)/60:.1f}min')

## Cell 11 — Build submission + Kaggle submit

In [ ]:
best = CHECKPOINT_DIR + '/final'
ac = best + '/adapter_config.json'; ab = best + '/adapter_model.safetensors'
assert os.path.exists(ac) and os.path.exists(ab), 'Adapter files missing'

cfg = json.load(open(ac))
# Verify Tong-style targets
assert 'lm_head' in cfg['target_modules'], 'lm_head required (train_unembed)'
assert 'in_proj' in cfg['target_modules']
assert cfg['r'] == 32

sz = SUBMISSIONS_DIR + '/v15-tong-replica.zip'
with zipfile.ZipFile(sz, 'w', zipfile.ZIP_DEFLATED) as z:
    z.write(ac, arcname='adapter_config.json')
    z.write(ab, arcname='adapter_model.safetensors')
    tc = best + '/tokenizer_config.json'
    if os.path.exists(tc): z.write(tc, arcname='tokenizer_config.json')

with open(sz, 'rb') as f: sha = hashlib.sha256(f.read()).hexdigest()
print(f'ZIP: {sz}  SHA: {sha[:12]}')

msg = f'v15 tong-replica step{gs} min-logprob-approx sha:{sha[:12]}'
r = subprocess.run(['kaggle', 'competitions', 'submit',
                    '-c', 'nvidia-nemotron-model-reasoning-challenge',
                    '-f', sz, '-m', msg],
                   capture_output=True, text=True, timeout=300)
print('Submit:', r.returncode, r.stdout, r.stderr[:200])

print('\nLB: https://www.kaggle.com/competitions/nvidia-nemotron-model-reasoning-challenge/submissions')